In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
import seaborn as sns

## load in damage data 

In [3]:
# load in damage files
path_ian_70_buildings = 'Hurricane_Ian_Shifted_70/impacts/Impacts_building_footprints_Hurricane_Ian_Shifted_70.gpkg'
all_building_damages = gpd.read_file(path_ian_70_buildings)

In [4]:
# select only the residential buildings
building_damages_res = all_building_damages.loc[all_building_damages['Primary Object Type'] == 'RES']

## load in EAD data 

In [6]:
impacts_detailed_path_geo = "data/current_risk_no_measures/Impacts/Impacts_building_footprints_current_risk_no_measures.gpkg"
impacts = gpd.read_file(impacts_detailed_path_geo)

DriverError: data/current_risk_no_measures/Impacts/Impacts_building_footprints_current_risk_no_measures.gpkg: No such file or directory

In [ ]:
# select only the residential buildings
impacts_res = impacts.loc[impacts['Primary Object Type'] == 'RES']

### Retrieve shapefiles of Charleston county

In [ ]:
# Access shapefile of State South Carolina
SC_bg = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/BG/tl_2021_45_bg.zip")

# Reproject shapefile to UTM Zone 17N
# https://spatialreference.org/ref/epsg/wgs-84-utm-zone-17n/
SC_bg = SC_bg.to_crs(epsg = 4326)

In [ ]:
# Drop multiple columns
# columns_to_drop = 
clean_SC_bg = SC_bg.drop(columns=['AWATER','MTFCC','ALAND','NAMELSAD','INTPTLAT','INTPTLON'])

#### Filter on Charleston county

In [ ]:
shape_charleston = clean_SC_bg[clean_SC_bg['COUNTYFP'] == '019']
shape_charleston['GEOID'] = shape_charleston['GEOID'].astype(int)
shape_charleston.info()

In [ ]:
# shape_charleston.to_file('data/shape_charleston.gpkg')

In [ ]:
# shape_charleston = gpd.read_file('data/shape_charleston.gpkg')

In [ ]:
# add geometry to floodadapt data by joining the files
damages_shape = building_damages_res.sjoin(shape_charleston, how='inner')
impacts_shape = impacts_res.sjoin(shape_charleston, how='inner')

In [ ]:
damages_shape['v_building']=damages_shape['Damage: Structure']/damages_shape['Max Potential Damage: Structure']
damages_shape['Risk (EAD)']=impacts_res['Risk (EAD)']

In [ ]:
damages_shape.info()

In [ ]:
plt.figure(figsize=(12, 5))  
fig = sns.histplot(data=damages_shape['v_building'], binwidth=0.02)
plt.xlabel('v building')
plt.xticks(np.arange(0, 1.05, 0.02)) 
plt.tick_params(axis='both', labelsize=5)  
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
plt.scatter(x=damages_shape.index, y=damages_shape['v_building'], s=1, alpha=0.6)
plt.axhline(y=0.03, color='red', linewidth=1)
plt.xlabel('Index')
plt.ylabel('v building')
plt.title('Scatterplot of v_building')
plt.xticks([])
plt.grid(True)

In [ ]:
damages_shape_hurricane = damages_shape[damages_shape['v_building'] >= 0.03]

In [ ]:
max_damage = {'Max Potential Damage: Structure': 'sum'}
damage_str = {'Damage: Structure':'sum'}
EAD = {'Risk (EAD)': 'sum'}

In [ ]:
number_buildings = damages_shape.groupby('GEOID').size()
number_buildings_hurricane = damages_shape_hurricane.groupby('GEOID').size()

# Convert group_sizes to a DataFrame
building_num_df = number_buildings.to_frame(name='building_per_bg')
building_num_df['building_per_bg_hurricane'] = number_buildings_hurricane
building_num_df.info()

In [ ]:
df_max_damage = damages_shape.groupby('GEOID').agg(max_damage)
df_max_damage_hurricane = damages_shape_hurricane.groupby('GEOID').agg(max_damage)
# df_max_damage.rename(columns={'GEOID': 'count'}, inplace=True)
# df_max_damage_hurricane.rename(columns={'GEOID': 'count'}, inplace=True)

In [ ]:
df_damage_str = damages_shape.groupby('GEOID').agg(damage_str)
df_damage_str_hurricane = damages_shape_hurricane.groupby('GEOID').agg(damage_str)
# df_damage_str.rename(columns={'GEOID': 'count'}, inplace=True)
# df_damage_str_hurricane.rename(columns={'GEOID': 'count'}, inplace=True)
df_damage_str_hurricane.head()

In [ ]:
df_EAD = damages_shape.groupby(damages_shape['GEOID']).aggregate(EAD)
df_EAD_hurricane = damages_shape_hurricane.groupby(damages_shape_hurricane['GEOID']).aggregate(EAD)

In [ ]:
data_frames = [df_max_damage, df_damage_str,df_EAD,building_num_df]
data_frames_hurricane = [df_max_damage_hurricane, df_damage_str_hurricane,df_EAD_hurricane,building_num_df]
df_damage = pd.concat(data_frames, axis=1, ignore_index=False, join='inner')
df_damage_hurricane = pd.concat(data_frames_hurricane, axis=1, ignore_index=False, join='inner')

In [ ]:
# Perform the join with shapefiles
agg_shape_damage = shape_charleston.merge(df_damage, on = "GEOID")
agg_shape_damage_hurricane = shape_charleston.merge(df_damage_hurricane, on = "GEOID")

In [ ]:
#further clean the dataset
agg_damage_clean = agg_shape_damage.drop(columns=['STATEFP','COUNTYFP','TRACTCE','BLKGRPCE','FUNCSTAT','building_per_bg','building_per_bg_hurricane'])
agg_damage_clean_hurricane = agg_shape_damage_hurricane.drop(columns=['STATEFP','COUNTYFP','TRACTCE','BLKGRPCE','FUNCSTAT','building_per_bg','building_per_bg_hurricane'])
agg_damage_clean.head(2)

In [ ]:
df_damage_hurricane

In [ ]:
agg_damage_clean_hurricane['ratio_damaged_buildings'] = df_damage_hurricane['building_per_bg_hurricane']/df_damage_hurricane['building_per_bg']

In [ ]:
agg_damage_clean.to_csv('data/damage_shape_bigfile.csv', index=False)
agg_damage_clean_hurricane.to_csv('data/damage_shape_hurricane.csv', index=False)

In [ ]:
(agg_damage_clean['Damage: Structure']/agg_damage_clean['Max Potential Damage: Structure']).mean()

In [ ]:
agg_damage_clean.head()

In [ ]:
agg_damage_clean.info()

In [ ]:
agg_damage_clean_hurricane.info()